In [63]:
#package imports
import pandas as pd

We use this notebook to perform an initial data merge. This assumes completion of the workflow contained in *core_data_workflow.ipynb*.

In [64]:
# loading in compustat, ceo data, ceo controls, our bridge (compustat/boardex id mapping), and our organized education data
financial_data = pd.read_csv("csv_data/compustat_large_cap_v2.csv")
ceo_data = pd.read_csv("csv_data/CEOS.csv")
ceo_controls = pd.read_csv("csv_data/CEO_CONTROLS.csv")
bridge = pd.read_csv("csv_data/boardex_id_data.csv")
education_organized = pd.read_csv("csv_data/EDUCATION_ORGANIZED.csv")

Our bridge should basically provide a link between our specified tickers and our boardid entries. Let's confirm that this is the case.

In [65]:
bridge.head(1)

,ticker,boardname,boardid
0,ATVI,ACTIVISION BLIZZARD INC (De-listed 10/2023),725


We need to start by linking our core financial data to this bridge. We merge our financial data with the bridge by matching ticker to "tic." This allows us to have a boardid for every ticker present in our dataset.

In [66]:
# perform basic left on merge
financial_data = financial_data.merge(
    bridge[['ticker', 'boardid']], 
    left_on='tic', 
    right_on='ticker', 
    how='left'
)
financial_data.drop(columns=['ticker'], inplace=True)

# visualize how the merge worked
financial_data.head(1)

,costat,curcd,datafmt,indfmt,consol,sic,datadate,gvkey,conm,tic,...,lse,ni,revt,xrd,csho,prcc_f,sich,mkt_cap,industry,boardid
0,A,USD,STD,INDL,C,3674,12/31/15,1161,ADVANCED MICRO DEVICES,AMD,...,3109.0,-660.0,3991.0,947.0,792.0,2.87,3674.0,2273.04,Semiconductors/Hardware,881.0


Given that this worked, we can inspect if we have any NaN values of boardid in our compustat dataset given that our number of boards ultimately went down.

In [67]:
financial_data["boardid"].isna().sum()

np.int64(364)

In [68]:
financial_data = financial_data.dropna(subset=['boardid'])
len(financial_data)

1201

To merge this with valuable CEO data, we must ensure that the date formatting of the start and end years is compatible with our fiscal year values.

We end up merging based on boardid = companyid (where companyid is the id of the company associated with a given ceo in boardex). This is similar to our first merge

In [69]:
# Ensure date columns are datetime
ceo_data['dateendrole'] = ceo_data['dateendrole'].replace('9000-01-01', '2027-01-01')

ceo_data['datestartrole'] = pd.to_datetime(ceo_data['datestartrole'], errors='coerce')
ceo_data['dateendrole'] = pd.to_datetime(ceo_data['dateendrole'], errors='coerce')

ceo_data['startyear'] = ceo_data['datestartrole'].dt.year
ceo_data['endyear'] = ceo_data['dateendrole'].dt.year

In [70]:
# Merge on boardid == companyid, then filter where fyear is strictly between start and end year
merged = financial_data.merge(
    ceo_data,
    left_on='boardid',
    right_on='companyid',
    how='left'
)

In [ ]:
# this eliminates overlapping ending years.
merged = merged[
    (merged['fyear'] >= merged['startyear']) &
    (merged['fyear'] < merged['endyear'])
]

# Drop helper columns 
merged = merged.drop(columns=['startyear', 'endyear'])

In [73]:
# we print the merged shape to see how things distributed out
print(merged.shape)
print(merged['gvkey'].nunique())
print(merged['fyear'].value_counts().sort_index())
print(merged['tic'].nunique())

(1097, 34)
161
fyear
2014     15
2015     88
2016     85
2017     90
2018     90
2019     90
2020    102
2021    114
2022    109
2023    112
2024    104
2025     98
Name: count, dtype: int64
161


We have healthy CEO-firm observations from 2015-2025 across 161 different firms represented. We will actually eliminate observations from 2014 by setting a year limit equal to and above 2015.

In [74]:
merged = merged[merged['fyear'] >= 2015]
print(merged.shape)

(1082, 34)


Let's also ensure we have a healthy distribution of observations across the different industries

In [75]:
print(merged['industry'].value_counts())
print(merged['gvkey'].nunique())

industry
Semiconductors/Hardware    418
Biotech                    400
Software                   264
Name: count, dtype: int64
160


Now we are in a position to merge in our education data given that we have our core CEO data in place.

In [76]:
# instigate left merge with education data to get in BVs for education
merged = merged.merge(
    education_organized,
    on='directorid',
    how='left'
)

In [77]:
# check that we have the correct columns
merged.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters'],
      dtype='object')

In [78]:
# we also have some CEO specific controls to add in place with a similar merge
merged = merged.merge(
    ceo_controls[['directorid', 'dob', 'gender', 'diversitynetworklabel']],
    on='directorid',
    how='left'
)
print(merged.shape)

(1082, 47)


In [79]:
# save our merge!
merged.to_csv("csv_data/MERGE1.csv", index=False)

In [80]:
merged.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel'],
      dtype='object')

We have successfully merged together our different merge objects into one MERGE1 csv file. We continue working with some additional data refinement steps in the *DataRefinement* folder and urge you to transition to the *data_refinement.ipynb* file to view additional steps in the refinement pipeline before performing regressions.